# 3. Integration and batch correction

The 8 samples are separate tissue cores: even on
one slide they differ in sectioning, cell density, and local signal. Before comparing them
we align their embeddings so that the same cell type from different samples overlaps.

### Setup

In [ ]:
import sys; sys.path.insert(0, '../scripts')
import numpy as np, pandas as pd
import scanpy as sc, squidpy as sq
import matplotlib.pyplot as plt
import course_utils as cu
sc.settings.verbosity = 1
plt.rcParams.update({'figure.facecolor': 'white', 'figure.dpi': 110})

## 1. Load all samples

In [ ]:
adata = cu.load_annotated()    # log-normalised, with .obs['celltype','sample','disease']
print(adata.shape)
adata.obs[['sample', 'disease']].value_counts().sort_index()

## 2. The batch effect, before correction

> **Method note.** Embed without correction
first. If cells separate by **sample** rather than by **cell type**, a batch effect is
driving the embedding. Some sample separation is expected here (different patients and
diseases), which is exactly why naive comparison is risky.

In [ ]:
sc.pp.pca(adata, n_comps=50)
sc.pp.neighbors(adata, n_neighbors=15)
sc.tl.umap(adata)
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
sc.pl.umap(adata, color=['sample', 'disease'], wspace=0.3)

## 3. Integrate with Harmony

> **Method note: what integration does, and its risk.**
Harmony iteratively clusters cells and removes the per-cluster shift between batches in PCA
space, fast and on CPU. Alternatives: **scVI/scANVI** (a deep generative model, better for
strong effects, needs a GPU) and Scanorama/BBKNN. The danger is **over-correction**: pushed
too hard, integration erases real biological differences between conditions, so always check
that cell types still separate and known markers stay where they should.

In [ ]:
# Harmony: a fast, CPU-friendly linear batch correction on the PCA embedding
sc.external.pp.harmony_integrate(adata, key='sample')   # -> obsm['X_pca_harmony']
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=1.0, flavor='igraph', n_iterations=2, directed=False, key_added='leiden_int')
sc.pl.umap(adata, color=['sample', 'celltype'], wspace=0.4)

> **Method note: did it mix?** A quick diagnostic: within each integrated cluster, measure
the entropy of the sample distribution. High entropy (near log of the number of samples)
means samples are well mixed; a cluster dominated by one sample is either unintegrated or a
genuinely sample-specific population.

In [ ]:
# simple mixing check: per integrated cluster, how evenly are samples mixed (entropy)
def sample_entropy(labels):
    p = labels.value_counts(normalize=True)
    return float(-(p * np.log(p)).sum())
ent = adata.obs.groupby('leiden_int', observed=True)['sample'].apply(sample_entropy)
print('max possible entropy:', round(np.log(adata.obs["sample"].nunique()), 2))
print(ent.round(2).sort_values().to_string())

## 4. Composition by disease

> **Method note.** With cells aligned, compare **cell-type
composition** across the disease groups. Treat this as descriptive: proportions shift with
tissue sampling, and here the groups are small (5 ANCA-GN, 1 GBM, 2 SLE). Notebook 4 tests
differences properly with the sample as the unit of replication.

In [ ]:
# cell-type composition per disease (the question integration sets up)
comp = pd.crosstab(adata.obs['disease'], adata.obs['celltype'], normalize='index')
comp.plot(kind='bar', stacked=True, figsize=(9, 4), colormap='tab20')
plt.ylabel('fraction of cells'); plt.legend(bbox_to_anchor=(1.02, 1), fontsize=7, ncol=1)
plt.title('Cell-type composition by disease'); plt.tight_layout(); plt.show()
comp.round(3)

> **Exercise.** Compute the immune-cell fraction per sample and compare ANCA-GN with the
other GN types. Is any apparent difference consistent across samples, or driven by one?

In [ ]:
# your code here


> **Exercise (ML).** A good integration should make a cell's **sample** hard to predict from
its embedding. Train a classifier to predict `sample` from the uncorrected PCA and from the
Harmony embedding, and compare cross-validated accuracy. Lower after integration means less
residual batch (but near-chance everywhere can also mean over-correction).

In [ ]:
# your code here


### Recap

We corrected sample/batch effects with Harmony, checked mixing, and read
composition by disease. Integration is necessary for fair comparison but can overcorrect;
verify it preserved biology. Next: test differences with proper statistics.